In [1]:
import pltkit
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import sys, os, glob, re
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.io.shapereader as shapereader
from matplotlib.ticker import FuncFormatter
sys.path.append(os.path.abspath(".."))
from matplotlib.patches import Rectangle

In [ ]:
"""
PARAMS
"""
wdir = 

## Figure 1

In [ ]:
# Calculate regional share of people over 65 years

region="IMAGE"
wdir = "X:/user/liprandicn/Projects/mt-comparison/models/carleton2022"
region_class = pd.read_csv(
        os.path.dirname(os.path.dirname(wdir)) +
        f"/data/RegionClassification/region_classification.csv"
        )[["hierid", "ISO3", "IMAGE26"]].iloc[:24378].rename(columns={"IMAGE26":"IMAGE"})
ir = gpd.read_file(wdir + "/data/CarletonSM/ir_shp/impact-region.shp")
ir["geometry"] = ir["geometry"].make_valid()

years = range(2010,2020)
pop_ssp = []

for age_group in ["young", "older", "oldest"]:
    pop_ssp_group = (
        pd.read_csv(wdir + f"/data/Population/PopulationIMAGE/pop_ssp2_{age_group}.csv")
        .pipe(
            lambda df: df.filter(
            ["hierid"] +
            [c for c in df.columns if c.isdigit() and int(c) in years]
        ))
        .set_index("hierid")
        .reindex(region_class["hierid"].values) # Align to impact regions orders
        .pipe(lambda df: df.set_axis(df.columns.astype(int), axis=1))
    )

    pop_ssp.append(pop_ssp_group.loc[:, years].to_numpy().astype(np.float32))
pop_ssp = np.stack(pop_ssp, axis=0)
pop_ssp = np.nanmean(pop_ssp, axis=-1)
ir["all"] = np.nansum(pop_ssp, axis=0)
ir["oldest"] = pop_ssp[-1]
ir = ir.merge(region_class, on="hierid")

ir = ir[["geometry", "all", "oldest", F"{region}"]]
ir = ir.dissolve(by="IMAGE", aggfunc=np.nansum)
#ir["share_oldest"] = ir["oldest"]/ir["all"]

In [ ]:
# Load regional values for insets

regions={"CHN":"China region", "INDIA":"India", "WEU":"Western Europe", "USA":"USA", "BRA":"Brazil", "SAF":"South Africa"}


wdir = "X:\\user\\liprandicn\\Projects\\mt-comparison\\models\\"
colors_list = ["#C8553D", "#566E3D", "#222E50", "#FEA82F", "#829191"]
temp_type = "heat"
age_group = "oldest"
cause = "All causes"
rt = "IMAGE"
var = "mortality"
years = range(2010,2020)

models = {
    "hon_file" : [
        0,
        "honda2014/output/ComparisonHonda/mortality_ComparisonHonda_SSP2_ERA5_1980-2023_counterfactual",
        "Honda et al., 2014",
        "H"
        ],
    "sco_file" : [
        1,
        "scovronick2024/output/ComparisonScovronick/mortality_ComparisonScovronick_SSP2_ERA5_1980-2023_counterfactual",
        "Scovronick et al., 2024",
        "S"
        ],
    "car_file" : [
        2,
        "carleton2022/output/ComparisonCarletonCounter/mortality_ComparisonCarletonCounter_SSP2_ERA5_NoAdap_1980-2023_*",
        "Carleton et al., 2022",
        "C"
        ],       
    "bur_file" : [
        3,
        "burkart2022/output/ComparisonBurkart/mortality_ComparisonBurkart_SSP2_ERA5_1980-2023_counterfactual_*",
        "Burkart et al., 2021",
        "B"
        ]
    }

region_vals={}
for region in regions:
    region_vals[region] = {"main": {}, "lower": {}, "upper": {}}
    for i,model in enumerate(models):
        m, l, u = pltkit.LoadMortalityDraws(wdir, models[model][1], rt, region, temp_type, cause, age_group, var, years)
        region_vals[region]["main"][model] = m.values
        region_vals[region]["lower"][model] = l.values
        region_vals[region]["upper"][model] = u.values
        
    region_vals[region]["name"] = regions[region]

In [38]:
# # Add population weighted GMST
# Run code only once

# wdir = "X:\\user\\liprandicn\\Projects\\mt-comparison\\"
# era5_dir = "X:/user/liprandicn/data/ERA5/t2m_daily"
# years=range(1980,2024)

# pop = xr.open_dataset(os.path.dirname(wdir) + f'/data/IMAGE/IMAGE_population/SSP2/GPOP.nc')
    
# # Reduce resolution to 15 min to match ERA5 data
# pop = pop.coarsen(latitude=3, longitude=3, boundary='pad').sum(skipna=True)

# # Select years
# pop = pop.sel(time=slice(f'{years[0]}-01-01', f'{years[-1]}-01-01'))

# temperaturas = [20, 25, 30, 35, 40]
# people_days = {tempe: [] for tempe in temperaturas}


# for year in years:
    
#     print(year)
#     # Read file and shift longitude coordinates
#     era5_daily = xr.open_dataset(era5_dir+f"/era5_t2m_mean_day_{year}.nc")

#     # Shift longitudinal coordinates  
#     era5_daily = era5_daily.assign_coords(longitude=((era5_daily.coords["longitude"] + 180) % 360 - 180)).sortby("longitude")

#     # Convert to Celsius 
#     era5_daily -= 273.15

#     era5_daily = era5_daily.interp(
#         latitude=np.clip(pop.latitude, era5_daily.latitude.min().item(), era5_daily.latitude.max().item()), 
#         method="nearest"
#         )

#     era5_daily=era5_daily.interp(longitude=pop.longitude, method="nearest")
#     era5_daily = era5_daily.drop_vars("number").t2m
    
#     pop_year = pop.sel(time=f"{year}-01-01").GPOP.drop_vars("time")
    
#     for tempe in [20,25,30,35,40]:
#         era_masked = (era5_daily > tempe) * pop_year
#         era_masked = era_masked.sum(dim=["latitude", "longitude", "valid_time"])
#         people_days[tempe].append(era_masked.item())
        
# df_anual = pd.DataFrame(people_days, index=years)

# df_anual["GPOP"] = pop.sum(dim=["latitude", "longitude"]).GPOP.values
# df_anual.to_csv("X:\\user\\liprandicn\\projects\\mt-comparison\\figures\\Paper1\\figure1_people_days_heat.csv")

In [39]:
def AddRegionalInset(ax_mapa, region, region_vals, coords, bounds_inset, xy_origen):
    
    ax_inset = ax_mapa.inset_axes(bounds_inset)
    
    fondo_extendido = Rectangle(
        (-0.30, -0.20), 1.4, 1.4, 
        transform=ax_inset.transAxes,
        facecolor='white',
        edgecolor='none',
        alpha=1.0,
        clip_on=False,                   
        zorder=3                         
    )

    ax_mapa.add_patch(fondo_extendido) 
    
    marco_delgado = Rectangle(
        (-0.30, -0.20), 1.4, 1.4,
        transform=ax_inset.transAxes,
        facecolor='none',             
        edgecolor='#b0b0b0',
        linewidth=0.3,    
        clip_on=False,
        zorder=4.5   
    )
    ax_mapa.add_patch(marco_delgado)
    
    ax_inset.patch.set_facecolor('whitesmoke')
    ax_inset.patch.set_alpha(1.0)
    ax_inset.set_facecolor("whitesmoke")
    ax_inset.set_zorder(4) 
    
    for model, color in zip(models, colors_list):
        err_minus = region_vals[region]["main"][model] - region_vals[region]["lower"][model]
        err_plus = region_vals[region]["upper"][model] - region_vals[region]["main"][model]
        
        ax_inset.errorbar(
            x=models[model][3], 
            y=region_vals[region]["main"][model], 
            yerr=[[err_minus], [err_plus]], 
            fmt='o',         
            capsize=1.5,       
            color=color, 
            ecolor=color,   
            markersize=2.5,
            linewidth=0.75
        )
    
    ax_inset.set_xlim(-0.5, len(models) - 0.5)
    ax_inset.grid(True, color="white", linewidth=0.5)
    ax_inset.yaxis.set_major_formatter(FuncFormatter(lambda x, pos: f'{x/1000:,.0f}k'))

    ax_inset.tick_params(axis='both', labelsize=4, which='both', length=0)
    for spine in ax_inset.spines.values():
        spine.set_visible(False)
        
    ax_inset.set_title(region_vals[region]["name"], fontsize=5, fontweight='bold')

    ax_mapa.annotate(
        '', 
        xy=coords,                                      
        xycoords=ccrs.PlateCarree()._as_mpl_transform(ax_mapa), 
        xytext=xy_origen,                                  
        textcoords=ax_inset.transAxes,                      
        arrowprops=dict(
            arrowstyle="-", 
            color="#444444", 
            lw=0.8,
            patchA=fondo_extendido, 
            shrinkA=0,              
            shrinkB=3               
        ),
        zorder=5               
    )

    ax_mapa.plot(
        coords[0], coords[1], 
        marker='o', 
        markersize=3,          
        color='#444444',       
        transform=ccrs.PlateCarree(),
        zorder=6      
    )
    
    return ax_inset

In [ ]:
fig = plt.figure(figsize=(5,6), dpi=300)

pos_ax1 = [0.1, 0.4, 0.85, 0.4]
pos_ax2 = [0.1, 0.05, 0.85, 0.25]
labelsize=6
titlesize=8
lettersize=7
years_vlines = [1998, 2010, 2016, 2019, 2022]

### -------------- Add upper left figure -------------
ax1 = fig.add_axes(pos_ax1)

for i, model in enumerate(models):
    main, lower, upper = pltkit.LoadMortalityDraws(wdir, models[model][1], rt, "World", temp_type, cause, age_group, var, None)
    
    line, = ax1.plot(main.year, main.values, label=models[model][2], linewidth=2, c=colors_list[i], clip_on=False)
    
    poly = ax1.fill_between(main.year, lower.values, upper.values, color=colors_list[i], alpha=0.3)
    poly.set_clip_on(False)

for year in years_vlines:
    ax1.axvline(x=year, color='gray', linestyle='--', linewidth=1, alpha=0.7)
ax1.set_title(f"Global heat mortality trends in the over-65 population", fontsize=titlesize, y=1.03)
ax1.legend(frameon=False, fontsize=6)
ax1.set_ylabel("Global excess mortality", fontsize=labelsize)
ax1.set_ylim(-0.9e5, 6.5e5)
ax1.tick_params(axis='both', labelsize=labelsize)


ax1.yaxis.set_major_formatter(FuncFormatter(lambda x, pos: f'{x/1000:g}k'))
ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)

ax1.text(-0.02, 1.07, 'A)', transform=ax1.transAxes, fontsize=lettersize, weight='bold', va='bottom', ha='right')


### ------------- Add lower figures (Separated vertically) -------------
df_anual = pd.read_csv("X:\\user\\liprandicn\\projects\\mt-comparison\\figures\\Paper1\\figure1_people_days_heat.csv", index_col=0)

temperatures = [25, 30, 35]
colors_reds = ["#FFC2C2", "#FF6B6B", "#C90000"]

# Parameters
x_start = pos_ax2[0]
y_start = pos_ax2[1]
width = pos_ax2[2]
total_height = pos_ax2[3]

num_plots = len(temperatures)
gap = 0.015 # Vetrical gap between subplots (in figure coordinates)
sub_height = (total_height - (gap * (num_plots - 1))) / num_plots

for i, t in enumerate(temperatures):
    # Get y position for the current subplot
    y_pos = y_start + (num_plots - 1 - i) * (sub_height + gap)
    
    # Create new axis
    ax_sub = fig.add_axes([x_start, y_pos, width, sub_height])
    
    # Plot data
    y_data = df_anual[f"{t}"] / df_anual["GPOP"]
    ax_sub.plot(df_anual.index, y_data, label=f"{t}°C", c=colors_reds[i], linewidth=2)
    
    for year in years_vlines:
        ax_sub.axvline(x=year, color='gray', linestyle='--', linewidth=1, alpha=0.7)
    
    # Configure axis
    ax_sub.spines["top"].set_visible(False)
    ax_sub.spines["right"].set_visible(False)
    ax_sub.tick_params(axis='both', labelsize=labelsize)
    ax_sub.set_ylabel(f"{t}°C", fontsize=6)

    
    # Add title only to the first subplot
    if i == 0:
        ax_sub.set_title("Annual individual exposure to heat over a temperature threshold", fontsize=titlesize, y=1.05)
        ax_sub.text(-0.02, 1.3, 'B)', transform=ax_sub.transAxes, fontsize=lettersize, weight='bold', va='bottom', ha='right')
        
    # Remove x-tick labels for all but the last subplot
    if i < num_plots - 1:
        ax_sub.set_xticklabels([])
        
plt.savefig(wdir+"figures\\Paper1\\figure1.png", dpi=300, bbox_inches='tight')
plt.savefig(wdir+"figures\\Paper1\\figure1.pdf", bbox_inches='tight')

In [ ]:
fig = plt.figure(figsize=(15,10), dpi=300)

pos_ax1 = [0.1, 0.5, 0.25, 0.25]
pos_ax2 = [0.1, 0.05, 0.35, 0.3]
pos_map = [0.40, 0.05, 0.60, 0.85]


### -------------- Add upper left figure -------------
ax1 = fig.add_axes(pos_ax1)

for i,model in enumerate(models):
    main, lower, upper = pltkit.LoadMortalityDraws(wdir, models[model][1], rt, "World", temp_type, cause, age_group, var, None)
    ax1.plot(main.year, main.values, label=models[model][2], linewidth=2, c=colors_list[i])
    ax1.fill_between(main.year, lower.values, upper.values, color=colors_list[i], alpha=0.3)


ax1.set_title(f"Historical climate-driven mortality from heat in people over 65", fontsize=8, y=1.03)
ax1.legend(frameon=False, fontsize=6)
ax1.set_ylabel("Excess mortality (thousand people)", fontsize=6)
ax1 = plt.gca()
ax1.yaxis.set_major_formatter(FuncFormatter(lambda x, pos: f'{x/1000:g}'))
ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)


### ------------- Add upper right figure -------------
df_anual = pd.read_csv(wdir+"figures\\Paper1\\figure1_people_days_heat.csv", index_col=0)
ax2 = fig.add_axes(pos_ax2)
ax2.set_title("People-days exposed tover a certain temperature", fontsize=8)
temperatures = [25, 30, 35]
colors_reds = ["#FFC2C2", "#FF6B6B", "#C90000", "#5C0000"]
for i,t in enumerate(temperatures):
    ax2.plot(df_anual.index, df_anual[f"{t}"]-df_anual[f"{t}"].iloc[0], label=f"{t}°C", c=colors_reds[i])
ax2.legend()


### --------------- Add map -------------
ax = fig.add_axes(pos_map, projection=ccrs.Robinson(central_longitude=0), frameon=True) 
ax.spines['geo'].set_linewidth(0.2)
ax.coastlines(resolution='10m', lw = 0.1)

im = ir.plot(
    ax=ax, 
    column="oldest",
    transform=ccrs.PlateCarree(), 
    cmap=colors.LinearSegmentedColormap.from_list(
            "custom_ramp", 
            ["#F8F9FA", "#B39EB5"], 
            N=256
        ),
    vmin=1e6
    # norm=colors.LogNorm(vmin=1e4, vmax=ir["oldest"].max())
    )
im = ax.collections[-1] 
ax.add_feature(cfeature.OCEAN, facecolor='white', zorder=2)

# Manually set colorbar limits
cbar = fig.colorbar(im, ax=ax, orientation='vertical', shrink=0.4, pad=0.05, aspect=20)
cbar.set_label('Number people over 65 years', fontsize=7)
cbar.ax.tick_params(labelsize=7, length=2, width=0.5)

# Add Antarctica 
land_shp = shapereader.natural_earth(resolution='110m', category='physical', name='land')
land_geoms = shapereader.Reader(land_shp).geometries()
for land in land_geoms:
    if land.bounds[1] < -60:
        ax.add_geometries([land], ccrs.PlateCarree(), 
                          facecolor='whitesmoke', 
                          edgecolor='none', 
                          zorder=2)
        
        
for reg, info in region_vals.items():
    AddRegionalInset(ax, "CHN", region_vals, coords=(104.19, 35.86), bounds_inset=[0.92, 0.65, 0.1, 0.25], xy_origen=(0,0.5))
    AddRegionalInset(ax, "INDIA", region_vals, coords=(77.20, 25), bounds_inset=[0.68, 0.2, 0.1, 0.25], xy_origen=(0.5,1))
    AddRegionalInset(ax, "WEU", region_vals, coords=(7, 48.85), bounds_inset=[0.35, 0.5, 0.1, 0.25], xy_origen=(1,1))
    AddRegionalInset(ax, "USA", region_vals, coords=(-103.77, 40), bounds_inset=[0.05, 0.55, 0.1, 0.25], xy_origen=(1,1))
    AddRegionalInset(ax, "BRA", region_vals, coords=(-50, -12.54), bounds_inset=[0.15, 0.15, 0.1, 0.25], xy_origen=(1,1))
    AddRegionalInset(ax, "SAF", region_vals, coords=(25, -28), bounds_inset=[0.4, 0.05, 0.1, 0.25], xy_origen=(1,1))

# plt.tight_layout()
plt.show()

## Figure 2

In [ ]:
region_type="IMAGE"
t_type = "heat"
variable="mortality"
age_group = "All ages"
region = "World"
years = range(2000,2101)
cause=None

scenarios = [ # "SSP1_L", "SSP1_M", "SSP1_ML",
  "SSP1_VLHO", "SSP1_VLLO",
  "SSP2_L", "SSP2_ML", "SSP2_M", #"SSP2_VLHO", "SSP2_VLLO",
  "SSP5_HL",
  "SSP3_H", #"SSP5_H", 
]

colours = ["#4A7C80", "#B89648", "#B56545", "#7E5E8F", "#557A42", "#5B6E91", "#AD526B", "#6E6E6E"]

fig, ax = plt.subplots(1,2, figsize=(16,8), dpi=300)
ax = ax.flatten()

for i,adap in enumerate(["_ssp", "_NoAdap"]):
  for j,scenario in enumerate(scenarios):
      filename = f"models/Carleton2022/output/ScenarioMIP7/mortality_ScenarioMIP7_{scenario}{adap}*"    
      mean, lower, upper = pltkit.LoadMortalityDraws(wdir, filename, region_type, region, t_type, cause, age_group, variable, years=None)
      ax[i].plot(mean.year, mean.values, linewidth=2, alpha=1, c=colours[j], label=scenario, clip_on=False)
      fill = ax[i].fill_between(mean.year, lower.values, upper.values, color=colours[j], alpha=0.1)
      fill.set_clip_on(False)
      
      # ax[0].set_ylim(-0.5e5, 22e5)
      ax[1].set_ylim(-0.5e6, 17e6)
      
      ax[0].set_ylabel("Global heat-related mortality (million people)", fontsize=13)
      ax[i].yaxis.set_major_formatter(FuncFormatter(lambda x, pos: f'{x/1e6:g}M'))
      ax[i].spines["top"].set_visible(False)
      ax[i].spines["right"].set_visible(False)
      
      ax[0].set_title("Full adaptation", fontsize=14)
      ax[1].set_title("No adaptation", fontsize=14)


leg = plt.legend(frameon=False, fontsize=12, bbox_to_anchor=(0.25,1))
for handle, text, color in zip(leg.legend_handles, leg.get_texts(), colours):
    text.set_color(color)
    # text.set_weight('bold')
    handle.set_visible(False)
    
# plt.suptitle("Heat-related mortality projections under ScenarioMIP7", y=1.01)
plt.show()

In [ ]:
region_type="IMAGE"
t_type = "heat"
variable="mortality"
age_group = "All ages"
region = "World"
years = range(2000,2101)
cause=None

scenarios = [
  "ssp245_r1", "ssp245_r2", "ssp245_r3", "ssp370_r1", "ssp370_r2", "ssp370_r3", "ssp585_r1", "ssp585_r2", "ssp585_r3"
]

colours = ["C0", "C1", "C2", "C3", "C4", "C5", "C6", "C7", "C8", "C9", "C10", "C11", "C12", "C13"]

for i,scenario in enumerate(scenarios):
    file_list = sorted(glob.glob(wdir+f"models/Carleton2022/output/ScenarioMIP7/mortality_ScenarioMIP7_SSP3_H_{scenario}*.nc"))
    for j in range(len(file_list)):
      filename = re.search(r'([^\\/]+)\.nc$', file_list[j]).group(1) 
      ds = pltkit.LoadMortality(wdir, filename, region_type, region, t_type, cause, age_group, variable)
      plt.plot(ds.year, ds.values, linewidth=0.4, c=colours[i], alpha=0.5)

plt.legend()
# plt.title("LHS 50 draws - SSP2_M_CP")
plt.show()

In [ ]:
region_type="IMAGE"
t_type = "heat"
variable="mortality"
age_group = "All ages"
region = "World"
years = range(2000,2101)
cause=None

scenarios = [
  "ssp245_r1", "ssp245_r2", "ssp245_r3", "ssp370_r1", "ssp370_r2", "ssp370_r3", "ssp585_r1", "ssp585_r2", "ssp585_r3"
]

colours = ["C0", "C1", "C2", "C3", "C4", "C5", "C6", "C7", "C8", "C9", "C10", "C11", "C12", "C13"]

for i,scenario in enumerate(scenarios):
    file_list = sorted(glob.glob(wdir+f"models/Carleton2022/output/ScenarioMIP7/mortality_ScenarioMIP7_SSP3_H_NoAdap_{scenario}_*.nc"))
    for j in range(len(file_list)):
      filename = re.search(r'([^\\/]+)\.nc$', file_list[j]).group(1) 
      ds = pltkit.LoadMortality(wdir, filename, region_type, region, t_type, cause, age_group, variable)
      plt.plot(ds.year, ds.values, linewidth=0.4, c=colours[i], alpha=0.5)

plt.legend()
# plt.title("LHS 50 draws - SSP2_M_CP")
plt.show()

## Figure 3